# Dr. Splat: Directly Referring 3D Gaussian Splatting
**CVPR 2025 Highlight**

1. 환경 설정
2. 데이터셋 다운로드 (LeRF-OVS)
3. 기본 3DGS 학습 (30K iterations)
4. Feature 추출 (SAM + OpenCLIP)
5. Dr. Splat Feature Registration
6. 시각화 (PCA / Activation)

---
## 0. Configuration

In [ ]:
import os, shutil, glob

WORK_DIR = "/content/drsplat"
MY_GITHUB_REPO = "https://github.com/BAEJUNHAK/Dr-Splat.git"

SCENE_NAME = "ramen"  # figurines / ramen / teatime / waldo_kitchen

DATASET_ROOT = f"{WORK_DIR}/data/lerf_ovs"
SCENE_PATH = f"{DATASET_ROOT}/{SCENE_NAME}"

DRSPLAT_REPO = f"{WORK_DIR}/Dr-Splat"
GS_REPO = f"{WORK_DIR}/gaussian-splatting"

GS_OUTPUT_PATH = f"{WORK_DIR}/output/3dgs_{SCENE_NAME}"
GS_CHECKPOINT = f"{GS_OUTPUT_PATH}/chkpnt30000.pth"
DRSPLAT_OUTPUT_PATH = f"{WORK_DIR}/output/drsplat_{SCENE_NAME}"

# Google Drive 백업 경로
DRIVE_SAVE_DIR = "/content/drive/MyDrive/drsplat_results"

GPU_ID = 0
TOPK = 40

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Scene: {SCENE_NAME}")
print(f"Scene path: {SCENE_PATH}")

---
## 1. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## 2. 환경 설정

### 2-1. 레포 클론

In [ ]:
# Dr-Splat (본인 GitHub) - .git 없으면 삭제 후 재클론
if os.path.exists(DRSPLAT_REPO) and os.path.isdir(os.path.join(DRSPLAT_REPO, '.git')):
    print("Dr-Splat exists. Pulling latest...")
    !cd {DRSPLAT_REPO} && git pull origin main
else:
    !rm -rf {DRSPLAT_REPO}
    !cd {WORK_DIR} && git clone {MY_GITHUB_REPO} --recursive
    print("Dr-Splat cloned!")

# 서브모듈 초기화 (중요!)
!cd {DRSPLAT_REPO} && git submodule update --init --recursive
print("\n=== submodules ===")
!ls {DRSPLAT_REPO}/submodules/

In [ ]:
# 원본 3DGS (기본 학습용)
if not os.path.exists(GS_REPO):
    !cd {WORK_DIR} && git clone https://github.com/graphdeco-inria/gaussian-splatting.git --recursive
    print("3DGS cloned!")
else:
    print("3DGS already exists.")

### 2-2. 패키지 설치
Colab 기본 PyTorch (CUDA 12.x)를 그대로 사용합니다.

In [ ]:
!pip install plyfile tqdm scipy scikit-image matplotlib opencv-python-headless -q
!pip install open-clip-torch faiss-cpu kmeans_pytorch -q
!pip install tensorboard tyro jaxtyping gdown -q

### 2-3. CUDA 서브모듈 빌드
> 빌드에 몇 분 걸릴 수 있습니다.

In [ ]:
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDA_PATH"] = "/usr/local/cuda"

# 원본 3DGS 서브모듈 (기본 학습용)
!pip install {GS_REPO}/submodules/diff-gaussian-rasterization
!pip install {GS_REPO}/submodules/simple-knn

---
## 3. Google Drive 마운트 & 데이터셋 다운로드
체크포인트 자동 저장/복원을 위해 먼저 Drive를 마운트합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Drive backup dir: {DRIVE_SAVE_DIR}")

### 3-1. LeRF-OVS 다운로드

| 소스 | 링크 |
|------|------|
| LangSplat Google Drive | [링크](https://drive.google.com/file/d/1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt/view) |
| OpenGaussian Baidu | [링크](https://pan.baidu.com/s/1B_tGYla5dWyJRu3jTNTMvA?pwd=u5iy) |

방법 1부터 시도하세요.

#### 방법 1: gdown (권장)

In [ ]:
import gdown, shutil

DOWNLOAD_DIR = f"{WORK_DIR}/data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

GDRIVE_ID = "1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt"
ZIP_PATH = f"{DOWNLOAD_DIR}/lerf_ovs.zip"

if not os.path.exists(SCENE_PATH):
    print("Downloading LeRF-OVS ...")
    try:
        gdown.download(id=GDRIVE_ID, output=ZIP_PATH, quiet=False, fuzzy=True)
        if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1_000_000:
            print(f"Download: {os.path.getsize(ZIP_PATH) / 1e9:.2f} GB")
            !cd {DOWNLOAD_DIR} && unzip -q -o {ZIP_PATH}
            !rm -f {ZIP_PATH}

            # 중첩 폴더 자동 처리: data/lerf_ovs/lerf_ovs/ -> data/lerf_ovs/
            nested = os.path.join(DATASET_ROOT, "lerf_ovs")
            if os.path.isdir(nested) and not os.path.isdir(os.path.join(DATASET_ROOT, SCENE_NAME)):
                tmp = f"{DOWNLOAD_DIR}/_lerf_tmp"
                shutil.move(nested, tmp)
                shutil.rmtree(DATASET_ROOT)
                shutil.move(tmp, DATASET_ROOT)
                print("Nested folder fixed!")
            print("Done!")
        else:
            print("[WARN] gdown 실패. 방법 2를 사용하세요.")
            !rm -f {ZIP_PATH}
    except Exception as e:
        print(f"[ERROR] {e}\n방법 2를 사용하세요.")
else:
    print(f"Dataset exists: {SCENE_PATH}")

#### 방법 2: Google Drive 마운트 (gdown 실패시)

In [ ]:
# === 주석 해제 후 실행 ===

# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ZIP = "/content/drive/MyDrive/lerf_ovs.zip"  # <-- 경로 수정
# !cp {DRIVE_ZIP} {DOWNLOAD_DIR}/
# !cd {DOWNLOAD_DIR} && unzip -q -o lerf_ovs.zip && rm lerf_ovs.zip

### 3-2. 데이터 구조 검증

In [ ]:
print("=== Available scenes ===")
if os.path.exists(DATASET_ROOT):
    for d in sorted(os.listdir(DATASET_ROOT)):
        full = os.path.join(DATASET_ROOT, d)
        if os.path.isdir(full):
            has_img = os.path.isdir(os.path.join(full, 'images'))
            has_sp = os.path.isdir(os.path.join(full, 'sparse', '0'))
            n = len(os.listdir(os.path.join(full, 'images'))) if has_img else 0
            print(f"  {d:20s} images:{n:3d}  sparse:{'OK' if has_sp else 'X'}")
else:
    print(f"[ERROR] {DATASET_ROOT} not found!")

def verify(path, name):
    checks = {
        'images/': os.path.isdir(os.path.join(path, 'images')),
        'sparse/0/cameras.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'cameras.bin')),
        'sparse/0/images.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'images.bin')),
        'sparse/0/points3D.bin': os.path.exists(os.path.join(path, 'sparse', '0', 'points3D.bin')),
    }
    ok = all(checks.values())
    print(f"\n{'PASS' if ok else 'FAIL'} - {name}")
    for k, v in checks.items():
        print(f"  {'[O]' if v else '[X]'} {k}")

verify(SCENE_PATH, SCENE_NAME)

### 3-3. Ref-Lerf 다운로드 (GT 평가용)
ReferSplat이 제공하는 GT 어노테이션 (test_json + gt_mask)을 다운로드합니다.
> 소스: [FudanCVL/Ref-Lerf (HuggingFace)](https://huggingface.co/datasets/FudanCVL/Ref-Lerf)

In [ ]:
!pip install huggingface_hub -q

REF_LERF_DIR = f"{WORK_DIR}/data/Ref-lerf"
REF_LERF_SCENE = f"{REF_LERF_DIR}/{SCENE_NAME}"

# Ref-lerf 테스트 JSON + mask 존재 확인
test_json_dir = os.path.join(REF_LERF_SCENE, "json", "test_json")
mask_dir = os.path.join(REF_LERF_SCENE, "mask")

if os.path.isdir(test_json_dir) and os.path.isdir(mask_dir):
    n_test = len([f for f in os.listdir(test_json_dir) if f.endswith('.json')])
    n_mask_dirs = len([f for f in os.listdir(mask_dir) if os.path.isdir(os.path.join(mask_dir, f))])
    print(f"Ref-lerf already exists: {REF_LERF_SCENE}")
    print(f"  test_json: {n_test} files, mask: {n_mask_dirs} frame dirs")
else:
    print("Downloading Ref-lerf from HuggingFace...")
    from huggingface_hub import hf_hub_download

    zip_path = hf_hub_download(
        repo_id="FudanCVL/Ref-Lerf",
        filename="Ref-lerf.zip",
        repo_type="dataset",
        local_dir=f"{WORK_DIR}/data",
    )
    print(f"Downloaded: {zip_path}")

    !cd {WORK_DIR}/data && unzip -q -o Ref-lerf.zip
    nested = os.path.join(REF_LERF_DIR, "Ref-lerf")
    if os.path.isdir(nested) and os.path.isdir(os.path.join(nested, SCENE_NAME)):
        import shutil
        tmp = f"{WORK_DIR}/data/_ref_tmp"
        shutil.move(nested, tmp)
        shutil.rmtree(REF_LERF_DIR)
        shutil.move(tmp, REF_LERF_DIR)
        print("Nested folder fixed!")

    print("Done!")

# 검증
print(f"\n=== Ref-lerf {SCENE_NAME} structure ===")
for sub in ["json/train_json", "json/test_json", "mask"]:
    p = os.path.join(REF_LERF_SCENE, sub)
    if os.path.isdir(p):
        files = os.listdir(p)
        print(f"  {sub}: {len(files)} items")
        for f in sorted(files)[:5]:
            print(f"    {f}")
    else:
        print(f"  {sub}: NOT FOUND")

### 3-3. SAM 체크포인트 다운로드

In [ ]:
SAM_CKPT = f"{DRSPLAT_REPO}/ckpts/sam_vit_h_4b8939.pth"
os.makedirs(f"{DRSPLAT_REPO}/ckpts", exist_ok=True)

if not os.path.exists(SAM_CKPT):
    print("Downloading SAM ViT-H...")
    !wget -q -P {DRSPLAT_REPO}/ckpts/ https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth
    print("Done!")
else:
    print("SAM checkpoint exists.")
!ls -lh {SAM_CKPT}

---
## 4. 기본 3DGS 학습 (30K iterations)
> T4 기준 약 20~40분 소요

In [ ]:
os.makedirs(GS_OUTPUT_PATH, exist_ok=True)

# Drive에서 체크포인트 확인
drive_gs_dir = os.path.join(DRIVE_SAVE_DIR, f"3dgs_{SCENE_NAME}")
drive_ckpt = os.path.join(drive_gs_dir, "chkpnt30000.pth")

if os.path.exists(GS_CHECKPOINT):
    # 로컬에 이미 있음
    print(f"Checkpoint exists (local): {GS_CHECKPOINT}")

elif os.path.exists(drive_ckpt):
    # Drive에서 복원
    print(f"Restoring checkpoint from Drive: {drive_gs_dir}")
    shutil.copytree(drive_gs_dir, GS_OUTPUT_PATH, dirs_exist_ok=True)
    print(f"Restored! {GS_CHECKPOINT}")

else:
    # 학습 실행
    print("Starting 3DGS training (30K iterations)...")
    !cd {GS_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python train.py \
        -s {SCENE_PATH} \
        -m {GS_OUTPUT_PATH} \
        --iterations 30000 \
        --save_iterations 30000 \
        --checkpoint_iterations 30000 \
        --eval
    print("Training done!")

    # 학습 완료 후 Drive에 자동 저장
    if os.path.exists(GS_CHECKPOINT):
        print(f"\nSaving checkpoint to Drive: {drive_gs_dir}")
        shutil.copytree(GS_OUTPUT_PATH, drive_gs_dir, dirs_exist_ok=True)
        print("Saved!")

In [ ]:
!ls -lh {GS_OUTPUT_PATH}/chkpnt*.pth 2>/dev/null || echo "No checkpoint found!"

---
## 5. Feature 추출 (SAM + OpenCLIP)
> **중요**: langsplat-rasterization으로 교체합니다.

In [ ]:
# 서브모듈 확인
!cd {DRSPLAT_REPO} && git submodule update --init --recursive
!ls {DRSPLAT_REPO}/submodules/

# 원본 rasterizer 제거 -> langsplat 버전 설치
!pip uninstall diff-gaussian-rasterization -y -q 2>/dev/null
!pip install {DRSPLAT_REPO}/submodules/langsplat-rasterization
!pip install {DRSPLAT_REPO}/submodules/segment-anything-langsplat

In [ ]:
# CLIP 레포
CLIP_DIR = f"{DRSPLAT_REPO}/CLIP"
if not os.path.exists(CLIP_DIR):
    !cd {DRSPLAT_REPO} && git clone https://github.com/openai/CLIP.git
else:
    print("CLIP exists.")

In [ ]:
LF_DIR = f"{SCENE_PATH}/language_features"
drive_lf_dir = os.path.join(DRIVE_SAVE_DIR, f"language_features_{SCENE_NAME}")

if os.path.exists(LF_DIR) and len(os.listdir(LF_DIR)) > 0:
    # 로컬에 이미 있음
    print(f"Language features exist (local): {LF_DIR}")
    !ls {LF_DIR} | head -10

elif os.path.exists(drive_lf_dir) and len(os.listdir(drive_lf_dir)) > 0:
    # Drive에서 복원
    print(f"Restoring language_features from Drive: {drive_lf_dir}")
    shutil.copytree(drive_lf_dir, LF_DIR, dirs_exist_ok=True)
    print(f"Restored! ({len(os.listdir(LF_DIR))} files)")

else:
    # 전처리 실행
    os.makedirs(LF_DIR, exist_ok=True)
    print("Starting feature extraction (SAM + OpenCLIP)...")
    print("This takes several hours on T4.")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python preprocessing.py \
        --dataset_path {SCENE_PATH} \
        --sam_ckpt_path {SAM_CKPT}
    print("Done!")

In [ ]:
import numpy as np
lf_files = sorted(os.listdir(LF_DIR))
f_files = [f for f in lf_files if f.endswith('_f.npy')]
print(f"Total: {len(lf_files)} files, {len(f_files)} features")
if f_files:
    print(f"Shape: {np.load(os.path.join(LF_DIR, f_files[0])).shape}")

In [ ]:
# language_features를 Drive에 백업 (전처리 5시간 날리지 않기 위해)
drive_lf_dir = os.path.join(DRIVE_SAVE_DIR, f"language_features_{SCENE_NAME}")
if os.path.exists(LF_DIR) and len(os.listdir(LF_DIR)) > 0:
    print(f"Saving language_features to Drive: {drive_lf_dir}")
    shutil.copytree(LF_DIR, drive_lf_dir, dirs_exist_ok=True)
    print(f"Saved! ({len(os.listdir(drive_lf_dir))} files)")
else:
    print("[WARN] language_features not found, nothing to save")

---
## 6. Dr. Splat Feature Registration
Majority Voting + PQ 압축 (gradient 최적화 아님)

In [ ]:
PQ_INDEX = f"{DRSPLAT_REPO}/ckpts/pq_index.faiss"
if os.path.exists(PQ_INDEX):
    print(f"PQ index: {PQ_INDEX}")
else:
    print("[ERROR] PQ index not found!")
    !ls -la {DRSPLAT_REPO}/ckpts/

In [ ]:
from argparse import Namespace
import faiss

# Dr. Splat 출력 경로 계산 (train.py가 suffix를 붙임)
_pq_index = faiss.read_index(f"{DRSPLAT_REPO}/ckpts/pq_index.faiss")
try:
    _suffix = f"_{1}_{('pq_openclip')}_topk{TOPK}_weight_{_pq_index.coarsecode_size()+_pq_index.code_size}"
except:
    _suffix = f"_{1}_{('pq_openclip')}_topk{TOPK}_weight_{_pq_index.code_size}"
DRSPLAT_ACTUAL_PATH = DRSPLAT_OUTPUT_PATH + _suffix
drive_drsplat_dir = os.path.join(DRIVE_SAVE_DIR, os.path.basename(DRSPLAT_ACTUAL_PATH))

# cfg_args 생성 함수
def ensure_cfg_args(model_path):
    cfg_path = os.path.join(model_path, "cfg_args")
    if not os.path.exists(cfg_path):
        os.makedirs(model_path, exist_ok=True)
        cfg = Namespace(
            sh_degree=3, source_path=SCENE_PATH, model_path=model_path,
            language_features_name='language_features_dim3', images='images',
            resolution=-1, white_background=False, feature_level=1,
            data_device='cuda', eval=True
        )
        with open(cfg_path, 'w') as f:
            f.write(str(cfg))
        print(f"cfg_args created: {cfg_path}")

# 로컬에 결과 있는지 확인
local_ckpt = os.path.join(DRSPLAT_ACTUAL_PATH, "chkpnt0.pth")
drive_ckpt = os.path.join(drive_drsplat_dir, "chkpnt0.pth")

if os.path.exists(local_ckpt):
    print(f"Dr. Splat result exists (local): {DRSPLAT_ACTUAL_PATH}")
    ensure_cfg_args(DRSPLAT_ACTUAL_PATH)

elif os.path.exists(drive_ckpt):
    print(f"Restoring Dr. Splat from Drive: {drive_drsplat_dir}")
    shutil.copytree(drive_drsplat_dir, DRSPLAT_ACTUAL_PATH, dirs_exist_ok=True)
    ensure_cfg_args(DRSPLAT_ACTUAL_PATH)
    print("Restored!")

else:
    print("Starting Dr. Splat feature registration...")
    !cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python train.py \
        -s {SCENE_PATH} \
        -m {DRSPLAT_OUTPUT_PATH} \
        --start_checkpoint {GS_CHECKPOINT} \
        --feature_level 1 \
        --name_extra pq_openclip \
        --use_pq \
        --pq_index {PQ_INDEX} \
        --port 55560 \
        --topk {TOPK} \
        --eval
    print("Done!")
    ensure_cfg_args(DRSPLAT_ACTUAL_PATH)

    # Drive에 자동 저장
    if os.path.exists(DRSPLAT_ACTUAL_PATH):
        print(f"Saving to Drive: {drive_drsplat_dir}")
        shutil.copytree(DRSPLAT_ACTUAL_PATH, drive_drsplat_dir, dirs_exist_ok=True)
        print("Saved!")

In [ ]:
DRSPLAT_TRAINED_PATH = DRSPLAT_ACTUAL_PATH
print(f"Dr. Splat output: {DRSPLAT_TRAINED_PATH}")
!ls -la {DRSPLAT_TRAINED_PATH}/ 2>/dev/null || echo "[WARN] Directory not found"

---
## 7. 시각화

### 7-1. PCA Feature Visualization

In [ ]:
!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_pca.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --pq_index {PQ_INDEX} \
    --feature_level 1 \
    -l language_features_dim3

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

pca_candidates = glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_pca", recursive=True)
if pca_candidates:
    pca_images = sorted(glob.glob(f"{pca_candidates[0]}/*.png"))[:8]
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    for ax, p in zip(axes.flat, pca_images):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis('off')
    plt.suptitle('PCA Feature Visualization', fontsize=16); plt.tight_layout(); plt.show()
else:
    print("PCA output not found")

### 7-2. Activation (텍스트 쿼리)

In [ ]:
TEXT_QUERY = "ramen"
SAVE_LABEL = "ramen_test"
THRESHOLD = 0.5

!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python render_activation.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --semantic_model sam \
    --feature_level 1 \
    --pq_index {PQ_INDEX} \
    --img_label "{TEXT_QUERY}" \
    --img_save_label "{SAVE_LABEL}" \
    --threshold {THRESHOLD} \
    -l language_features_dim3

In [ ]:
# 결과 시각화: colormap / binary / side-by-side
for viz_type in ["colormap", "binary", "sidebyside"]:
    candidates = glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_{viz_type}_{SAVE_LABEL}", recursive=True)
    if not candidates:
        continue
    images = sorted(glob.glob(f"{candidates[0]}/*.png"))[:4]
    if not images:
        continue
    fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 5))
    if len(images) == 1:
        axes = [axes]
    for ax, p in zip(axes, images):
        ax.imshow(Image.open(p)); ax.set_title(os.path.basename(p), fontsize=8); ax.axis('off')
    plt.suptitle(f'{viz_type}: "{TEXT_QUERY}" (thr={THRESHOLD})', fontsize=14)
    plt.tight_layout(); plt.show()

if not any(glob.glob(f"{DRSPLAT_TRAINED_PATH}/**/renders_*_{SAVE_LABEL}", recursive=True)):
    print("No activation images found.")

---## 8. 정량 평가 실행

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

REF_LERF_DIR = f"{WORK_DIR}/data/Ref-lerf"
EVAL_OUTPUT = os.path.join(WORK_DIR, "eval_results")
CUSTOM_QUERIES = f"{DRSPLAT_REPO}/queries_ramen.json"

# Level 0 (category) + Level 5 (sentence) 는 --ref_lerf_dir 에서 자동 평가
# Level 1, 4 는 --custom_queries 로 추가 평가
!cd {DRSPLAT_REPO} && CUDA_VISIBLE_DEVICES={GPU_ID} python evaluate_drsplat.py \
    -s {SCENE_PATH} \
    -m {DRSPLAT_TRAINED_PATH} \
    --pq_index {PQ_INDEX} \
    --ref_lerf_dir {REF_LERF_DIR} \
    --output_dir {EVAL_OUTPUT} \
    --custom_queries {CUSTOM_QUERIES} \
    --threshold 0.5 \
    -l language_features_dim3


In [ ]:
RESULTS_JSON = os.path.join(EVAL_OUTPUT, SCENE_NAME, "results.json")

if not os.path.exists(RESULTS_JSON):
    print(f"[ERROR] {RESULTS_JSON} not found! Section 8의 evaluate_drsplat.py를 먼저 실행하세요.")
    ALL_RESULTS = []
else:
    with open(RESULTS_JSON) as f:
        ALL_RESULTS = json.load(f)
    print(f"Total evaluations: {len(ALL_RESULTS)}")

# 레벨 매핑: query_type -> level label
LEVEL_MAP = {"category": "L0", "sentence": "L5"}

for r in ALL_RESULTS:
    qt = r.get("query_type", "")
    r["level"] = qt if qt.startswith("L") else LEVEL_MAP.get(qt, qt)


---
## 9. 레벨별 정량 평가 결과


### 9-0. 전체 요약


In [ ]:
if not ALL_RESULTS:
    print("[SKIP] 결과 없음")
else:
    levels = sorted(set(r["level"] for r in ALL_RESULTS))

    header = f"{'Level':<8s} {'mIoU':>8s} {'mAcc@.25':>10s} {'Precision':>10s} {'Recall':>8s} {'F1':>8s} {'N':>5s}"
    print(header)
    print("-" * 60)

    for lv in levels:
        lv_results = [r for r in ALL_RESULTS if r["level"] == lv]
        ious = [r["iou"] for r in lv_results]
        acc25 = [r["acc_25"] for r in lv_results]
        prec = [r["precision"] for r in lv_results]
        rec = [r["recall"] for r in lv_results]
        f1s = [r["f1"] for r in lv_results]
        print(f"{lv:<8s} {np.mean(ious):>7.2f}% {np.mean(acc25)*100:>9.2f}% {np.mean(prec):>9.2f}% {np.mean(rec):>7.2f}% {np.mean(f1s):>7.2f}% {len(lv_results):>5d}")

    # 전체
    ious_all = [r["iou"] for r in ALL_RESULTS]
    acc25_all = [r["acc_25"] for r in ALL_RESULTS]
    print("-" * 60)
    print(f"{'ALL':<8s} {np.mean(ious_all):>7.2f}% {np.mean(acc25_all)*100:>9.2f}%{'  ':>10s}{'  ':>8s}{'  ':>8s} {len(ALL_RESULTS):>5d}")

    # 바 차트
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    x = range(len(levels))
    palette = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0", "#F44336"]
    miou_vals = [np.mean([r["iou"] for r in ALL_RESULTS if r["level"] == lv]) for lv in levels]
    macc_vals = [np.mean([r["acc_25"] for r in ALL_RESULTS if r["level"] == lv]) * 100 for lv in levels]

    ax1.bar(x, miou_vals, color=palette[:len(levels)])
    ax1.set_xticks(list(x)); ax1.set_xticklabels(levels)
    ax1.set_ylabel("mIoU (%)"); ax1.set_title("mIoU by Level")
    for idx, v in enumerate(miou_vals):
        ax1.text(idx, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)

    ax2.bar(x, macc_vals, color=palette[:len(levels)])
    ax2.set_xticks(list(x)); ax2.set_xticklabels(levels)
    ax2.set_ylabel("mAcc@0.25 (%)"); ax2.set_title("mAcc@0.25 by Level")
    for idx, v in enumerate(macc_vals):
        ax2.text(idx, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)

    plt.suptitle(f"Dr. Splat Level-wise Performance - {SCENE_NAME}", fontsize=14)
    plt.tight_layout(); plt.show()


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

def show_level_visualization(all_results, level, scene_name, scene_path):
    """RGB | GT mask (검은배경+흰마스크) | Pred mask (검은배경+흰마스크)"""
    lv_results = [r for r in all_results if r["level"] == level]
    if not lv_results:
        print(f"[SKIP] {level} 결과 없음")
        return

    by_cat_frame = defaultdict(dict)
    for r in lv_results:
        by_cat_frame[r["category"]][r["frame"]] = r

    cats = sorted(by_cat_frame.keys())

    for cat in cats:
        cat_frames = by_cat_frame[cat]
        frames_sorted = sorted(cat_frames.keys())
        n = len(frames_sorted)
        if n == 0:
            continue

        fig, axes = plt.subplots(n, 3, figsize=(15, 4 * n))
        if n == 1:
            axes = [axes]

        query_text = ""
        for i, frame in enumerate(frames_sorted):
            r = cat_frames[frame]
            query_text = r.get("query_text", cat)
            iou = r["iou"]

            # RGB
            orig_img = None
            for ext in [".png", ".jpg", ".JPG"]:
                p = os.path.join(scene_path, "images", frame + ext)
                if os.path.exists(p):
                    orig_img = np.array(Image.open(p))
                    break
            if orig_img is not None:
                axes[i][0].imshow(orig_img)
            axes[i][0].set_title(f"{frame}", fontsize=9)
            axes[i][0].axis("off")

            # GT mask: 검은 배경 + 흰색
            gt_path = r.get("gt_path", "")
            if os.path.exists(gt_path):
                gt = cv2.imread(gt_path, cv2.IMREAD_GRAYSCALE)
                gt_vis = np.zeros((*gt.shape, 3), dtype=np.uint8)
                gt_vis[gt > 127] = [255, 255, 255]
                axes[i][1].imshow(gt_vis)
            axes[i][1].set_title("GT Mask", fontsize=9)
            axes[i][1].axis("off")

            # Pred mask: 검은 배경 + 흰색
            pred_path = r.get("pred_path", "")
            if os.path.exists(pred_path):
                pred = cv2.imread(pred_path, cv2.IMREAD_GRAYSCALE)
                pred_vis = np.zeros((*pred.shape, 3), dtype=np.uint8)
                pred_vis[pred > 127] = [255, 255, 255]
                axes[i][2].imshow(pred_vis)
            axes[i][2].set_title(f"Pred (IoU={iou:.1f}%)", fontsize=9)
            axes[i][2].axis("off")

        q_label = query_text if len(query_text) <= 60 else query_text[:57] + "..."
        fig.suptitle(f"[{level}] \"{q_label}\" → GT: {cat}", fontsize=12, fontweight="bold")
        plt.tight_layout()
        plt.show()


### 9-1. Level 0: Base Retrieval (GT 카테고리 단어)


In [ ]:
if not ALL_RESULTS:
    print("[SKIP]")
else:
    lv_results = [r for r in ALL_RESULTS if r["level"] == "L0"]
    if not lv_results:
        print("[SKIP] L0 결과 없음")
    else:
        cat_data = defaultdict(list)
        for r in lv_results:
            cat_data[r["category"]].append(r)

        frames = sorted(set(r["frame"] for r in lv_results))
        frame_labels = [f[-3:] for f in frames]

        # 헤더
        header = "Category".ljust(20) + "".join(fl.rjust(8) for fl in frame_labels) + "mIoU".rjust(8) + "Acc@.25".rjust(8)
        print(header)
        print("-" * len(header))

        cats_sorted = sorted(cat_data.keys(), key=lambda c: np.mean([r["iou"] for r in cat_data[c]]), reverse=True)
        for cat in cats_sorted:
            rs = cat_data[cat]
            frame_ious = {r["frame"]: r["iou"] for r in rs}
            row = cat.ljust(20)
            for f in frames:
                if f in frame_ious:
                    row += f"{frame_ious[f]:>7.1f}%"
                else:
                    row += "      --"
            miou = np.mean([r["iou"] for r in rs])
            acc = np.mean([r["acc_25"] for r in rs]) * 100
            row += f"{miou:>7.2f}%{acc:>7.0f}%"
            print(row)

        print("-" * len(header))
        total_miou = np.mean([r["iou"] for r in lv_results])
        total_acc = np.mean([r["acc_25"] for r in lv_results]) * 100
        pad = " " * (len(frame_labels) * 8)
        total_label = "TOTAL".ljust(20)
        print(f"{total_label}{pad}{total_miou:>7.2f}%{total_acc:>7.0f}%")

        # 카테고리별 바 차트
        fig, ax = plt.subplots(figsize=(14, 5))
        miou_vals = [np.mean([r["iou"] for r in cat_data[c]]) for c in cats_sorted]
        colors = ["#4CAF50" if v >= 25 else "#F44336" for v in miou_vals]
        ax.barh(range(len(cats_sorted)), miou_vals, color=colors)
        ax.set_yticks(range(len(cats_sorted))); ax.set_yticklabels(cats_sorted)
        ax.set_xlabel("mIoU (%)"); ax.set_title(f"Level 0 (Base Retrieval) - {SCENE_NAME}")
        ax.axvline(x=25, color="red", linestyle="--", alpha=0.5, label="Acc@0.25 threshold")
        ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
show_level_visualization(ALL_RESULTS, "L0", SCENE_NAME, SCENE_PATH)


### 9-2. Level 1: 단어/속성 변화


In [ ]:
if not ALL_RESULTS:
    print("[SKIP]")
else:
    lv_results = [r for r in ALL_RESULTS if r["level"] == "L1"]
    if not lv_results:
        print("[SKIP] L1 결과 없음")
    else:
        l0_results = [r for r in ALL_RESULTS if r["level"] == "L0"]
        l0_by_cat = defaultdict(list)
        for r in l0_results:
            l0_by_cat[r["category"]].append(r["iou"])

        frames = sorted(set(r["frame"] for r in lv_results))
        frame_labels = [f[-3:] for f in frames]

        l1_by_cat = defaultdict(list)
        l1_query = {}
        for r in lv_results:
            l1_by_cat[r["category"]].append(r)
            l1_query[r["category"]] = r["query_text"]

        header = "Category".ljust(20) + "".join(fl.rjust(8) for fl in frame_labels) + "mIoU".rjust(8) + "L0".rjust(8) + "Delta".rjust(8) + "  Query"
        print(header)
        print("-" * len(header))

        cats = sorted(l1_by_cat.keys())
        for cat in cats:
            rs = l1_by_cat[cat]
            frame_ious = {r["frame"]: r["iou"] for r in rs}
            row = cat.ljust(20)
            for f in frames:
                if f in frame_ious:
                    row += f"{frame_ious[f]:>7.1f}%"
                else:
                    row += "      --"
            l1_miou = np.mean([r["iou"] for r in rs])
            l0_miou = np.mean(l0_by_cat[cat]) if cat in l0_by_cat else 0
            delta = l1_miou - l0_miou
            sign = "+" if delta >= 0 else ""
            row += f"{l1_miou:>7.2f}%{l0_miou:>7.2f}% {sign}{delta:>6.2f}%  {l1_query[cat]}"
            print(row)

        print("-" * len(header))
        l1_mean = np.mean([r["iou"] for r in lv_results])
        l0_mean = np.mean([r["iou"] for r in l0_results]) if l0_results else 0
        d = l1_mean - l0_mean
        pad = " " * (len(frame_labels) * 8)
        sign = "+" if d >= 0 else ""
        mean_label = "MEAN".ljust(20)
        print(f"{mean_label}{pad}{l1_mean:>7.2f}%{l0_mean:>7.2f}% {sign}{d:>6.2f}%")

        # 비교 차트
        fig, ax = plt.subplots(figsize=(14, 6))
        x = np.arange(len(cats))
        w = 0.35
        l0_vals = [np.mean(l0_by_cat[c]) if c in l0_by_cat else 0 for c in cats]
        l1_vals = [np.mean([r["iou"] for r in l1_by_cat[c]]) for c in cats]
        ax.bar(x - w/2, l0_vals, w, label="L0 (category)", color="#4CAF50")
        ax.bar(x + w/2, l1_vals, w, label="L1 (attribute)", color="#2196F3")
        ax.set_xticks(x); ax.set_xticklabels(cats, rotation=45, ha="right")
        ax.set_ylabel("mIoU (%)"); ax.set_title(f"Level 0 vs Level 1 - {SCENE_NAME}")
        ax.legend(); ax.axhline(y=25, color="red", linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()


In [ ]:
show_level_visualization(ALL_RESULTS, "L1", SCENE_NAME, SCENE_PATH)


### 9-2b. Level 2: 세부 파트 (정성 평가)


In [ ]:
# Level 2: 배치 렌더링 (GT 없음, activation binary만)
import sys, time, copy
sys.path.insert(0, DRSPLAT_REPO)

L2_QUERIES = ['bowl rim', 'chopstick tip', 'egg yolk', 'egg white', 'glass rim',
              'fingertips', 'kamaboko swirl', 'nori edge', 'plate rim', 'cup rim',
              'spoon handle', 'noodle strand', 'water', 'bottle']

_orig_cwd = os.getcwd()
os.chdir(DRSPLAT_REPO)

from render_activation import render_batch
from evaluation.openclip_encoder import OpenCLIPNetwork
import faiss
from argparse import Namespace

try:
    clip_model_cache
except NameError:
    clip_model_cache = OpenCLIPNetwork("cuda")
    index_cache = faiss.read_index(PQ_INDEX)

l2_pairs = [(q, q.replace(" ", "_")[:40]) for q in L2_QUERIES]

fake_args = Namespace(
    source_path=SCENE_PATH, model_path=DRSPLAT_TRAINED_PATH,
    sh_degree=3, images="images", resolution=-1, white_background=False,
    data_device="cuda", eval=True, feature_level=1,
    language_features_name="language_features_dim3", threshold=0.5,
    include_feature=False, convert_SHs_python=False,
    compute_cov3D_python=False, debug=False,
)
parser = __import__("argparse").ArgumentParser()
from arguments import ModelParams, PipelineParams
model_params = ModelParams(parser, sentinel=True)
pipeline_params = PipelineParams(parser)
dataset = model_params.extract(fake_args)
pipe = pipeline_params.extract(fake_args)

print(f"=== Level 2: {len(L2_QUERIES)} queries ===")
print(f"Output dir: {DRSPLAT_TRAINED_PATH}/test/")
render_batch(dataset, pipe, fake_args, l2_pairs, clip_model_cache, index_cache, threshold=0.5)
print("Done!")

os.chdir(_orig_cwd)


In [ ]:
# Level 2 시각화: RGB | Pred
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

L2_QUERIES = ['bowl rim', 'chopstick tip', 'egg yolk', 'egg white', 'glass rim',
              'fingertips', 'kamaboko swirl', 'nori edge', 'plate rim', 'cup rim',
              'spoon handle', 'noodle strand', 'water', 'bottle']

for query in L2_QUERIES:
    save_label = query.replace(" ", "_")[:40]
    binary_dir = os.path.join(DRSPLAT_TRAINED_PATH, "test", f"renders_binary_{save_label}")
    if not os.path.isdir(binary_dir):
        print(f'[SKIP] "{query}" - not found: {binary_dir}')
        continue
    binary_images = sorted(glob.glob(os.path.join(binary_dir, "*.png")))[:7]
    if not binary_images:
        print(f'[SKIP] "{query}" - no images in {binary_dir}')
        continue

    # side-by-side도 확인 (RGB 포함)
    sbs_dir = os.path.join(DRSPLAT_TRAINED_PATH, "test", f"renders_sidebyside_{save_label}")

    n = len(binary_images)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = [axes]

    for i, bp in enumerate(binary_images):
        # Binary 이미지 읽기 (컬러: 빨강=활성, 회색=비활성)
        pred_img = cv2.imread(bp)
        if pred_img is not None:
            pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
            axes[i][1].imshow(pred_img)
        axes[i][1].set_title("Pred (red=active)", fontsize=9)
        axes[i][1].axis("off")

        # RGB: side-by-side 첫 1/3 사용, 없으면 colormap 사용
        fname = os.path.basename(bp)
        sbs_path = os.path.join(sbs_dir, fname)
        if os.path.exists(sbs_path):
            sbs_img = cv2.imread(sbs_path)
            sbs_img = cv2.cvtColor(sbs_img, cv2.COLOR_BGR2RGB)
            w = sbs_img.shape[1] // 3
            axes[i][0].imshow(sbs_img[:, :w])
        axes[i][0].set_title(f"RGB ({fname})", fontsize=9)
        axes[i][0].axis("off")

    fig.suptitle(f'[L2] "{query}"', fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()


### 9-2c. Level 3: 같은 종류 구분 (정성 평가)


In [ ]:
# Level 3: 배치 렌더링
L3_QUERIES = ['left chopstick', 'left egg', 'both eggs', 'right nori']

_orig_cwd = os.getcwd()
os.chdir(DRSPLAT_REPO)

l3_pairs = [(q, q.replace(" ", "_")[:40]) for q in L3_QUERIES]

print(f"=== Level 3: {len(L3_QUERIES)} queries ===")
render_batch(dataset, pipe, fake_args, l3_pairs, clip_model_cache, index_cache, threshold=0.5)
print("Done!")

os.chdir(_orig_cwd)


In [ ]:
# Level 3 시각화: RGB | Pred
L3_QUERIES = ['left chopstick', 'left egg', 'both eggs', 'right nori']

for query in L3_QUERIES:
    save_label = query.replace(" ", "_")[:40]
    binary_dir = os.path.join(DRSPLAT_TRAINED_PATH, "test", f"renders_binary_{save_label}")
    if not os.path.isdir(binary_dir):
        print(f'[SKIP] "{query}" - not found: {binary_dir}')
        continue
    binary_images = sorted(glob.glob(os.path.join(binary_dir, "*.png")))[:7]
    if not binary_images:
        print(f'[SKIP] "{query}" - no images in {binary_dir}')
        continue

    sbs_dir = os.path.join(DRSPLAT_TRAINED_PATH, "test", f"renders_sidebyside_{save_label}")

    n = len(binary_images)
    fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))
    if n == 1:
        axes = [axes]

    for i, bp in enumerate(binary_images):
        pred_img = cv2.imread(bp)
        if pred_img is not None:
            pred_img = cv2.cvtColor(pred_img, cv2.COLOR_BGR2RGB)
            axes[i][1].imshow(pred_img)
        axes[i][1].set_title("Pred (red=active)", fontsize=9)
        axes[i][1].axis("off")

        fname = os.path.basename(bp)
        sbs_path = os.path.join(sbs_dir, fname)
        if os.path.exists(sbs_path):
            sbs_img = cv2.imread(sbs_path)
            sbs_img = cv2.cvtColor(sbs_img, cv2.COLOR_BGR2RGB)
            w = sbs_img.shape[1] // 3
            axes[i][0].imshow(sbs_img[:, :w])
        axes[i][0].set_title(f"RGB ({fname})", fontsize=9)
        axes[i][0].axis("off")

    fig.suptitle(f'[L3] "{query}"', fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()


### 9-3. Level 4: 공간 관계


In [ ]:
if not ALL_RESULTS:
    print("[SKIP]")
else:
    lv_results = [r for r in ALL_RESULTS if r["level"] == "L4"]
    if not lv_results:
        print("[SKIP] L4 결과 없음")
    else:
        l0_results = [r for r in ALL_RESULTS if r["level"] == "L0"]
        l0_by_cat = defaultdict(list)
        for r in l0_results:
            l0_by_cat[r["category"]].append(r["iou"])

        frames = sorted(set(r["frame"] for r in lv_results))
        frame_labels = [f[-3:] for f in frames]

        lx_by_cat = defaultdict(list)
        lx_query = {}
        for r in lv_results:
            lx_by_cat[r["category"]].append(r)
            lx_query[r["category"]] = r["query_text"]

        header = "Category".ljust(20) + "".join(fl.rjust(8) for fl in frame_labels) + "mIoU".rjust(8) + "L0".rjust(8) + "Delta".rjust(8) + "  Query"
        print(header)
        print("-" * len(header))

        cats = sorted(lx_by_cat.keys())
        for cat in cats:
            rs = lx_by_cat[cat]
            frame_ious = {r["frame"]: r["iou"] for r in rs}
            row = cat.ljust(20)
            for f in frames:
                if f in frame_ious:
                    row += f"{frame_ious[f]:>7.1f}%"
                else:
                    row += "      --"
            lx_miou = np.mean([r["iou"] for r in rs])
            l0_miou = np.mean(l0_by_cat[cat]) if cat in l0_by_cat else 0
            delta = lx_miou - l0_miou
            sign = "+" if delta >= 0 else ""
            row += f"{lx_miou:>7.2f}%{l0_miou:>7.2f}% {sign}{delta:>6.2f}%  {lx_query[cat]}"
            print(row)

        print("-" * len(header))
        lx_mean = np.mean([r["iou"] for r in lv_results])
        l0_mean = np.mean([r["iou"] for r in l0_results]) if l0_results else 0
        d = lx_mean - l0_mean
        pad = " " * (len(frame_labels) * 8)
        sign = "+" if d >= 0 else ""
        mean_label = "MEAN".ljust(20)
        print(f"{mean_label}{pad}{lx_mean:>7.2f}%{l0_mean:>7.2f}% {sign}{d:>6.2f}%")

        # 비교 차트
        fig, ax = plt.subplots(figsize=(14, 6))
        x = np.arange(len(cats))
        w = 0.35
        l0_vals = [np.mean(l0_by_cat[c]) if c in l0_by_cat else 0 for c in cats]
        lx_vals = [np.mean([r["iou"] for r in lx_by_cat[c]]) for c in cats]
        ax.bar(x - w/2, l0_vals, w, label="L0 (category)", color="#4CAF50")
        ax.bar(x + w/2, lx_vals, w, label="L4 (spatial)", color="#FF9800")
        ax.set_xticks(x); ax.set_xticklabels(cats, rotation=45, ha="right")
        ax.set_ylabel("mIoU (%)"); ax.set_title(f"Level 0 vs L4 - {SCENE_NAME}")
        ax.legend(); ax.axhline(y=25, color="red", linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()


In [ ]:
show_level_visualization(ALL_RESULTS, "L4", SCENE_NAME, SCENE_PATH)


### 9-4. Level 5: 설명문


In [ ]:
if not ALL_RESULTS:
    print("[SKIP]")
else:
    lv_results = [r for r in ALL_RESULTS if r["level"] == "L5"]
    if not lv_results:
        print("[SKIP] L5 결과 없음")
    else:
        l0_results = [r for r in ALL_RESULTS if r["level"] == "L0"]
        l0_by_cat = defaultdict(list)
        for r in l0_results:
            l0_by_cat[r["category"]].append(r["iou"])

        frames = sorted(set(r["frame"] for r in lv_results))
        frame_labels = [f[-3:] for f in frames]

        lx_by_cat = defaultdict(list)
        lx_query = {}
        for r in lv_results:
            lx_by_cat[r["category"]].append(r)
            lx_query[r["category"]] = r["query_text"][:50]

        header = "Category".ljust(20) + "".join(fl.rjust(8) for fl in frame_labels) + "mIoU".rjust(8) + "L0".rjust(8) + "Delta".rjust(8) + "  Query (truncated)"
        print(header)
        print("-" * len(header))

        cats = sorted(lx_by_cat.keys())
        for cat in cats:
            rs = lx_by_cat[cat]
            frame_ious = {r["frame"]: r["iou"] for r in rs}
            row = cat.ljust(20)
            for f in frames:
                if f in frame_ious:
                    row += f"{frame_ious[f]:>7.1f}%"
                else:
                    row += "      --"
            lx_miou = np.mean([r["iou"] for r in rs])
            l0_miou = np.mean(l0_by_cat[cat]) if cat in l0_by_cat else 0
            delta = lx_miou - l0_miou
            sign = "+" if delta >= 0 else ""
            row += f"{lx_miou:>7.2f}%{l0_miou:>7.2f}% {sign}{delta:>6.2f}%  {lx_query[cat]}"
            print(row)

        print("-" * len(header))
        lx_mean = np.mean([r["iou"] for r in lv_results])
        l0_mean = np.mean([r["iou"] for r in l0_results]) if l0_results else 0
        d = lx_mean - l0_mean
        pad = " " * (len(frame_labels) * 8)
        sign = "+" if d >= 0 else ""
        mean_label = "MEAN".ljust(20)
        print(f"{mean_label}{pad}{lx_mean:>7.2f}%{l0_mean:>7.2f}% {sign}{d:>6.2f}%")

        # 비교 차트
        fig, ax = plt.subplots(figsize=(14, 6))
        x = np.arange(len(cats))
        w = 0.35
        l0_vals = [np.mean(l0_by_cat[c]) if c in l0_by_cat else 0 for c in cats]
        lx_vals = [np.mean([r["iou"] for r in lx_by_cat[c]]) for c in cats]
        ax.bar(x - w/2, l0_vals, w, label="L0 (category)", color="#4CAF50")
        ax.bar(x + w/2, lx_vals, w, label="L5 (sentence)", color="#9C27B0")
        ax.set_xticks(x); ax.set_xticklabels(cats, rotation=45, ha="right")
        ax.set_ylabel("mIoU (%)"); ax.set_title(f"Level 0 vs L5 - {SCENE_NAME}")
        ax.legend(); ax.axhline(y=25, color="red", linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()


In [ ]:
show_level_visualization(ALL_RESULTS, "L5", SCENE_NAME, SCENE_PATH)


---
## Troubleshooting

| 문제 | 해결 |
|------|------|
| **OOM** | `TOPK`를 10~20으로 줄이기 |
| **submodules 없음** | `git submodule update --init --recursive` 실행 |
| **rasterizer 충돌** | 5단계에서 uninstall -> langsplat 설치 확인 |
| **PQ index 없음** | `ckpts/pq_index.faiss` 확인 |
| **런타임 재시작 후 데이터 없음** | Drive 백업에서 복원 또는 재다운로드 |